# 07_eval / cross_dataset_eval.ipynb

**Purpose:** Formal **out-of-distribution** evaluation of the GSAF fusion model on:

1. **MMFakeBench test** (held-out — neither train nor val_tune)
2. **VERITE test** (real-world OOC pairs, never seen during training)

This is the most important table for the paper because it shows the model generalizes beyond the training distribution.

**Inputs:**
- `cache/fusion/fusion_predictions.csv`
- `cache/fusion/fusion_thresholds.json`
- Per-stream prediction CSVs in `cache/stream_*/`
- `data/processed/ooc_pairs/manifests/all.csv` (fallback for dataset/split)

**Outputs:**
- `paper/tables/cross_dataset_eval_mmfakebench.csv`
- `paper/tables/cross_dataset_eval_verite.csv`
- `paper/tables/cross_dataset_eval_combined.csv` — long-form, both datasets, all models
- `paper/figures/cross_dataset_eval_auroc_bars.png`
- `paper/figures/cross_dataset_eval_f1_bars.png`


In [ ]:
# Imports & path setup
from __future__ import annotations
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score, classification_report,
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

NB_DIR = Path.cwd()
ROOT = NB_DIR
for parent in [NB_DIR] + list(NB_DIR.parents):
    if (parent / "src" / "utils" / "paths.py").exists():
        ROOT = parent
        break
sys.path.insert(0, str(ROOT))

from src.utils.paths import (
    ALL_MANIFEST, FUSION_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, STREAM_C_CACHE,
    PAPER, FIGURES, TABLES,
)

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print("Repo root:", ROOT)


## 1. Load fusion predictions + thresholds

In [ ]:
fusion_path = FUSION_CACHE / "fusion_predictions.csv"
thresh_path = FUSION_CACHE / "fusion_thresholds.json"
assert fusion_path.exists(), f"Missing {fusion_path}"

fusion_df = pd.read_csv(fusion_path)
print("fusion_predictions shape:", fusion_df.shape)

if thresh_path.exists():
    thresholds = json.loads(thresh_path.read_text())
else:
    thresholds = {}

print("Thresholds:", thresholds)


In [ ]:
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

ds_col    = find_col(fusion_df, ["dataset", "source_dataset", "src_dataset"])
split_col = find_col(fusion_df, ["split", "data_split"])
y_col     = find_col(fusion_df, ["y_true", "label", "target"])

if ds_col is None or split_col is None:
    manifest = pd.read_csv(ALL_MANIFEST, usecols=lambda c: c in {"pair_id", "dataset", "split"})
    fusion_df = fusion_df.merge(manifest, on="pair_id", how="left")
    ds_col, split_col = "dataset", "split"

print("Dataset col:", ds_col, "| split col:", split_col, "| y col:", y_col)
print("Counts per (dataset, split):")
print(fusion_df.groupby([ds_col, split_col]).size().head(20))


## 2. Metric helper + filter helper

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, name="model"):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    mask = ~np.isnan(y_prob) & ~np.isnan(y_true)
    y_true, y_prob = y_true[mask], y_prob[mask]
    if len(np.unique(y_true)) < 2:
        return {"model": name, "n": int(mask.sum()), "auroc": np.nan, "auprc": np.nan,
                "accuracy": np.nan, "f1": np.nan, "precision": np.nan, "recall": np.nan,
                "threshold": threshold}
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "model": name, "n": int(mask.sum()),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "threshold": threshold,
    }

CANDIDATE_COLS = {
    "GSAF (full)":              ["fusion_prob", "prob_fusion", "y_prob_fusion", "y_prob"],
    "Ablation: no Stream A":    ["prob_no_a", "ab_no_a_prob", "y_prob_no_a"],
    "Ablation: no Stream C":    ["prob_no_c", "ab_no_c_prob", "y_prob_no_c"],
    "Ablation: MLP fusion":     ["prob_mlp_fusion", "ab_mlp_prob", "y_prob_mlp"],
    "Ablation: frozen CLIP":    ["prob_frozen_clip", "ab_frozen_prob", "y_prob_frozen_clip"],
    "Ablation: no MLLM":        ["prob_no_mllm", "ab_no_mllm_prob", "y_prob_no_mllm"],
}

STREAM_FILES = {
    "Stream A (entities)":  STREAM_A_CACHE / "results" / "stream_a_predictions.csv",
    "Stream B (CLIP)":      STREAM_B_CACHE / "final"   / "stream_b_predictions.csv",
    "Stream C (NLI)":       STREAM_C_CACHE / "results" / "stream_c_predictions.csv",
}

def evaluate_dataset(dataset_name, split_name, thresh_key):
    """Return DataFrame of metrics for fusion + ablations + per-stream baselines on a (dataset, split)."""
    threshold = float(thresholds.get(thresh_key, thresholds.get("default", 0.5)))
    sub = fusion_df[(fusion_df[ds_col] == dataset_name) & (fusion_df[split_col] == split_name)].copy()
    print(f"\n{dataset_name}/{split_name}: {len(sub):,} rows  (threshold={threshold:.4f})")
    rows = []

    # Fusion + ablations
    for label, cands in CANDIDATE_COLS.items():
        col = find_col(sub, cands)
        if col is None:
            continue
        rows.append(compute_metrics(sub[y_col], sub[col], threshold=threshold, name=label))

    # Streams (own files)
    for name, fp in STREAM_FILES.items():
        if not fp.exists():
            continue
        sdf = pd.read_csv(fp)
        sds  = find_col(sdf, ["dataset", "source_dataset"])
        ssp  = find_col(sdf, ["split", "data_split"])
        sy   = find_col(sdf, ["y_true", "label", "target"])
        sp   = find_col(sdf, ["y_prob", "prob", "score", "stream_prob"])
        if sds is None or ssp is None:
            manifest = pd.read_csv(ALL_MANIFEST, usecols=lambda c: c in {"pair_id", "dataset", "split"})
            sdf = sdf.merge(manifest, on="pair_id", how="left")
            sds, ssp = "dataset", "split"
        ssub = sdf[(sdf[sds] == dataset_name) & (sdf[ssp] == split_name)]
        if len(ssub) == 0 or sy is None or sp is None:
            continue
        rows.append(compute_metrics(ssub[sy], ssub[sp], threshold=0.5, name=name))

    df = pd.DataFrame(rows)
    return df[["model", "n", "auroc", "auprc", "accuracy", "f1", "precision", "recall", "threshold"]]


## 3. MMFakeBench test

In [ ]:
mmfb_metrics = evaluate_dataset("mmfakebench", "test", thresh_key="mmfakebench_test")
mmfb_metrics.to_csv(TABLES / "cross_dataset_eval_mmfakebench.csv", index=False, float_format="%.4f")
print("Saved:", TABLES / "cross_dataset_eval_mmfakebench.csv")
mmfb_metrics


## 4. VERITE test

In [ ]:
# VERITE test split — name may be 'test' or 'verite_test' depending on manifest convention
verite_metrics = evaluate_dataset("verite", "test", thresh_key="verite_test")
if verite_metrics.empty:
    print("Empty for split='test'; trying split='verite_test'…")
    verite_metrics = evaluate_dataset("verite", "verite_test", thresh_key="verite_test")

verite_metrics.to_csv(TABLES / "cross_dataset_eval_verite.csv", index=False, float_format="%.4f")
print("Saved:", TABLES / "cross_dataset_eval_verite.csv")
verite_metrics


## 5. Combined long-form table

In [ ]:
mmfb_metrics["dataset"] = "MMFakeBench-test"
verite_metrics["dataset"] = "VERITE-test"
combined = pd.concat([mmfb_metrics, verite_metrics], ignore_index=True)
combined = combined[["dataset", "model", "n", "auroc", "auprc", "accuracy", "f1", "precision", "recall", "threshold"]]
combined.to_csv(TABLES / "cross_dataset_eval_combined.csv", index=False, float_format="%.4f")
print("Saved:", TABLES / "cross_dataset_eval_combined.csv")
combined.style.format({c: "{:.4f}" for c in ["auroc","auprc","accuracy","f1","precision","recall","threshold"]})


## 6. Bar plots: AUROC and F1 by dataset

In [ ]:
def plot_metric(metric, fname, ylabel):
    pivot = combined.pivot(index="model", columns="dataset", values=metric)
    pivot = pivot.reindex(combined["model"].drop_duplicates().tolist())  # preserve order
    pivot = pivot.dropna(how="all")
    fig, ax = plt.subplots(figsize=(max(8, 1.0 * len(pivot)), 5))
    pivot.plot(kind="bar", ax=ax, width=0.75)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_title(f"Cross-dataset {ylabel} (out-of-distribution)")
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)
    ax.set_xticklabels(pivot.index, rotation=20, ha="right")
    ax.legend(title="", loc="lower right")
    plt.tight_layout()
    out = FIGURES / fname
    plt.savefig(out, bbox_inches="tight", dpi=150)
    print("Saved:", out)
    plt.show()

plot_metric("auroc", "cross_dataset_eval_auroc_bars.png", "AUROC")
plot_metric("f1",    "cross_dataset_eval_f1_bars.png",    "F1")


## 7. Headline numbers (for the paper abstract)

In [ ]:
def headline(df, dataset_name):
    full = df[df["model"] == "GSAF (full)"]
    if full.empty:
        print(f"[{dataset_name}] GSAF (full) row not found.")
        return
    r = full.iloc[0]
    print(f"[{dataset_name}]  AUROC={r['auroc']:.4f} | AUPRC={r['auprc']:.4f} | F1={r['f1']:.4f} | n={int(r['n'])}")

headline(mmfb_metrics, "MMFakeBench-test")
headline(verite_metrics, "VERITE-test")


---
Notes:
- All thresholds are tuned on validation, never on test.
- If a stream's prediction file does not contain rows for VERITE/MMFakeBench, that row is silently dropped.
- Compare GSAF vs the strongest individual stream and the MLP-fusion ablation to argue the gating mechanism's contribution.